In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parents[0]))
print(sys.path[-1])

/mnt/D/graph_ssl/Graph-SSL


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import networkx as nx

from torch_geometric.utils import to_networkx
from torch_geometric.datasets import Planetoid
from torch_geometric.transforms import NormalizeFeatures

from wl_gcl.src.utils.wl_core import WLHierarchyEngine
from wl_gcl.src.models.base_gnn import GINEncoder  # start with this


/home/farouc/envs/base/lib/python3.12/site-packages/torch/__config__.py:10: UserWarning: CUDA initialization: CUDA unknown error - this may be due to an incorrectly set up environment, e.g. changing env variable CUDA_VISIBLE_DEVICES after program start. Setting the available devices to be zero. (Triggered internally at ../c10/cuda/CUDAFunctions.cpp:108.)
  return torch._C._show_config()


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

dataset = Planetoid(root="/tmp/Cora", name="Cora", transform=NormalizeFeatures())
data = dataset[0].to(device)

N = data.num_nodes

In [4]:
# Build WL on CPU-friendly structures
edge_index_cpu = data.edge_index.cpu()
edges = list(zip(edge_index_cpu[0].tolist(), edge_index_cpu[1].tolist()))
nodes = list(range(N))

engine = WLHierarchyEngine(nodes, edges)
engine.build_wl_tree(max_iterations=10)  # will early-stop

# Determine existing WL levels (global)
all_levels = sorted({t for v in nodes for t in engine.node_path[v].keys()})
print("WL levels:", all_levels)

Building WL Tree (Convergence Mode: False)...
-> Stable convergence reached at iteration 5.
WL levels: [0, 1, 2, 3, 4, 5]


In [5]:
level_targets = {}
for t in all_levels:
    if t == 0:
        continue
    y, cid2idx, C = engine.get_level_targets(t)
    if y is not None:
        level_targets[t] = {
            "y": y.to(device),
            "cid2idx": cid2idx,
            "num_classes": C
        }

print({t: level_targets[t]["num_classes"] for t in level_targets})


{1: 37, 2: 1589, 3: 2301, 4: 2363, 5: 2365}


In [6]:
from wl_gcl.src.models.wl_multilevel import WLMultilevelModel
from wl_gcl.src.losses.wl_losses import (
    wl_classification_loss,
    hierarchy_regularization
)


In [7]:
in_dim = data.x.size(1)
hidden_dim = 64
out_dim = 64

encoder = GINEncoder(input_dim=in_dim, hidden_dim=hidden_dim, output_dim=out_dim, num_layers=3).to(device)
model = WLMultilevelModel(encoder, out_dim, level_targets).to(device)

opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)


In [8]:
sorted_levels = sorted(level_targets.keys())  # e.g. [1,2,3,...,T]

def levels_for_epoch(epoch, warmup=20):
    # gradually include deeper levels
    k = min(len(sorted_levels), 1 + epoch // max(1, warmup // len(sorted_levels)))
    return sorted_levels[:k]


In [9]:
lambda_hier = 1e-2  # tune later
epochs = 50

for epoch in range(epochs):
    model.train()
    opt.zero_grad()

    z, logits = model(data.x, data.edge_index)

    active_levels = levels_for_epoch(epoch, warmup=20)

    # WL classification loss
    loss_wl = 0.0
    for t in active_levels:
        y_t = level_targets[t]["y"]
        loss_wl = loss_wl + F.cross_entropy(logits[t], y_t)

    # hierarchy regularization
    # use levels 1..max active (needs consecutive levels for t-1)
    max_t = max(active_levels)
    reg_levels = [t for t in range(1, max_t + 1) if t in sorted_levels]
    loss_hier = hierarchy_regularization(engine, z, reg_levels)

    loss = loss_wl + lambda_hier * loss_hier
    loss.backward()
    opt.step()

    if epoch % 5 == 0:
        print(f"epoch={epoch:03d} levels={active_levels} loss={loss.item():.4f} "
              f"wl={loss_wl.item():.4f} hier={loss_hier.item():.4f}")


epoch=000 levels=[1] loss=3.5809 wl=3.5809 hier=0.0000
epoch=005 levels=[1, 2] loss=10.7853 wl=10.7816 hier=0.3710
epoch=010 levels=[1, 2, 3] loss=18.3882 wl=18.3850 hier=0.3205
epoch=015 levels=[1, 2, 3, 4] loss=26.0090 wl=26.0062 hier=0.2734
epoch=020 levels=[1, 2, 3, 4, 5] loss=33.6143 wl=33.6119 hier=0.2447
epoch=025 levels=[1, 2, 3, 4, 5] loss=33.4428 wl=33.4403 hier=0.2455
epoch=030 levels=[1, 2, 3, 4, 5] loss=33.2679 wl=33.2653 hier=0.2615
epoch=035 levels=[1, 2, 3, 4, 5] loss=33.0924 wl=33.0896 hier=0.2828
epoch=040 levels=[1, 2, 3, 4, 5] loss=32.9149 wl=32.9119 hier=0.2988
epoch=045 levels=[1, 2, 3, 4, 5] loss=32.7362 wl=32.7331 hier=0.3136


In [10]:
import torch.nn.functional as F

def cos(a, b):
    return F.cosine_similarity(a, b, dim=0).item()


In [11]:
def pick_valid_level(engine, u):
    levels = sorted(engine.node_path[u].keys())
    for t in levels[::-1]:  # start from deep, go up
        cluster = engine.get_cluster_at_level(u, t)
        if cluster is not None and len(cluster) > 1:
            return t
    return None


In [12]:
u = 5
t = pick_valid_level(engine, u)
print("Using level:", t)

pos = engine.get_cluster_at_level(u, t)
neg = list(set(range(N)) - set(pos))

pos_scores = [cos(z[u], z[v]) for v in pos if v != u][:20]
neg_scores = [cos(z[u], z[v]) for v in neg][:20]

print("pos mean:", sum(pos_scores)/len(pos_scores),
      "neg mean:", sum(neg_scores)/len(neg_scores))


Using level: 2
pos mean: 0.7652181200683117 neg mean: 0.31333842780441046


## Phase 2: Curriculum learning with WL enhanced InfoNCE

In [13]:
from wl_gcl.src.models.wl_multilevel import WLMultilevelModel
from wl_gcl.src.losses.wl_losses import (
    hierarchy_regularization,
    wl_contrastive_loss
)


In [14]:
edge_index_cpu = data.edge_index.cpu()
edges = list(zip(edge_index_cpu[0].tolist(), edge_index_cpu[1].tolist()))
nodes = list(range(N))

engine = WLHierarchyEngine(nodes, edges)
engine.build_wl_tree(max_iterations=10)

all_levels = sorted({t for v in nodes for t in engine.node_path[v].keys()})
print("WL levels:", all_levels)


Building WL Tree (Convergence Mode: False)...
-> Stable convergence reached at iteration 5.
WL levels: [0, 1, 2, 3, 4, 5]


In [15]:
in_dim = data.x.size(1)
hidden_dim = 64
out_dim = 64

encoder = GINEncoder(in_dim, hidden_dim, out_dim, num_layers=3).to(device)
model = WLMultilevelModel(encoder, out_dim, level_targets).to(device)

opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)


In [16]:
def levels_for_epoch(epoch, warmup=20):
    k = min(len(sorted_levels), 1 + epoch // max(1, warmup // len(sorted_levels)))
    return sorted_levels[:k]


In [17]:
#Training with WL + Hierarchy + WL-NCE

lambda_hier = 1e-2
lambda_nce  = 1e-1
epochs = 50

for epoch in range(epochs):
    model.train()
    opt.zero_grad()

    z, logits = model(data.x, data.edge_index)

    active_levels = levels_for_epoch(epoch)

    # ---- WL classification ----
    loss_wl = torch.tensor(0.0, device=device)
    for t in active_levels:
        y_t = level_targets[t]["y"]
        loss_wl += F.cross_entropy(logits[t], y_t)

    # ---- hierarchy regularization ----
    max_t = max(active_levels)
    reg_levels = [t for t in range(1, max_t + 1) if t in sorted_levels]
    loss_hier = hierarchy_regularization(engine, z, reg_levels)

    # ---- WL contrastive ----
    t_contrast = max(active_levels)
    loss_nce = wl_contrastive_loss(engine, z, level=t_contrast)

    loss = loss_wl + lambda_hier * loss_hier + lambda_nce * loss_nce
    loss.backward()
    opt.step()

    if epoch % 5 == 0:
        print(
            f"epoch={epoch:03d} levels={active_levels} "
            f"loss={loss.item():.4f} "
            f"wl={loss_wl.item():.4f} "
            f"hier={loss_hier.item():.4f} "
            f"nce={loss_nce.item():.4f}"
        )


epoch=000 levels=[1] loss=3.8334 wl=3.6202 hier=0.0000 nce=2.1316
epoch=005 levels=[1, 2] loss=12.5680 wl=10.8585 hier=0.3820 nce=17.0564
epoch=010 levels=[1, 2, 3] loss=20.9185 wl=18.4706 hier=0.4221 nce=24.4362
epoch=015 levels=[1, 2, 3, 4] loss=28.6135 wl=26.1040 hier=0.4410 nce=25.0510
epoch=020 levels=[1, 2, 3, 4, 5] loss=36.1998 wl=33.6965 hier=0.4546 nce=24.9874
epoch=025 levels=[1, 2, 3, 4, 5] loss=36.0171 wl=33.5225 hier=0.4699 nce=24.8994
epoch=030 levels=[1, 2, 3, 4, 5] loss=35.8373 wl=33.3491 hier=0.4840 nce=24.8337
epoch=035 levels=[1, 2, 3, 4, 5] loss=35.6577 wl=33.1732 hier=0.4891 nce=24.7964
epoch=040 levels=[1, 2, 3, 4, 5] loss=35.4787 wl=32.9965 hier=0.4906 nce=24.7725
epoch=045 levels=[1, 2, 3, 4, 5] loss=35.3053 wl=32.8244 hier=0.4899 nce=24.7597


In [18]:
model.eval()
with torch.no_grad():
    z, _ = model(data.x, data.edge_index)

u = 5
t = pick_valid_level(engine, u)

pos = engine.get_cluster_at_level(u, t)
neg = list(set(range(N)) - set(pos))[:200]

pos_scores = [cos(z[u], z[v]) for v in pos if v != u][:20]
neg_scores = [cos(z[u], z[v]) for v in neg][:20]

print("level:", t)
print("pos mean:", sum(pos_scores)/len(pos_scores))
print("neg mean:", sum(neg_scores)/len(neg_scores))


level: 2
pos mean: 0.9267134703695774
neg mean: 0.5764780826866627
